In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 22.6 MB/s eta 0:00:00


In [ ]:
!pip install openai

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Step 1: Create a Document Index with FAISS
def create_faiss_index(documents, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    # Convert documents to embeddings
    embedder = SentenceTransformer(model_name)
    embeddings = embedder.encode(documents, convert_to_tensor=False)

    # Create FAISS index
    dimension = embeddings.shape[1]  # Embedding dimensions
    index = faiss.IndexFlatL2(dimension)  # L2 similarity
    index.add(embeddings)
    return index, embedder

# Step 2: Retrieve Relevant Documents
def retrieve_documents(query, index, embedder, documents, top_k=3):
    query_embedding = embedder.encode([query], convert_to_tensor=False)
    distances, indices = index.search(query_embedding, top_k)
    return [documents[i] for i in indices[0]]

# Step 3: Use T5 Model for Response Augmentation
def generate_response_t5(query, retrieved_docs, model_name="google/flan-t5-large"): # Change model_name to a sequence-to-sequence model
    # Load T5 tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).cuda()  # Use GPU (cuda:0)

    # Prepare the input prompt
    context = " ".join(retrieved_docs)
    print(context)
    input_text = (
        f"context: {context} \n\n"
        f"Question: {query} \n\n"
        f"Provide a detailed answer based on the most relevant information by Explain in detail:"
    )
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True  # Explicitly truncate the input
    )

    # Generate the response
    outputs = model.generate(
        inputs.input_ids.cuda(),  # Move inputs to GPU
        max_length=150,
        num_beams=20,
        early_stopping=True
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Example Usage
if __name__ == "__main__":
    # Sample documents
    documents = [
        "Python is a popular programming language known for its simplicity.",
        "FAISS is a library for efficient similarity search and clustering of dense vectors.",
        "T5 is a model developed by Google for various NLP tasks like summarization and Q&A."
    ]

    # Create FAISS index
    index, embedder = create_faiss_index(documents)

    # Query input
    query = "what is python?"

    # Retrieve documents
    retrieved_docs = retrieve_documents(query, index, embedder, documents)
    print("Retrieved Documents:", retrieved_docs)

    # Generate response using T5
    response = generate_response_t5(query, retrieved_docs)
    print("Answer:", response)


Retrieved Documents: ['Python is a popular programming language known for its simplicity.', 'T5 is a model developed by Google for various NLP tasks like summarization and Q&A.', 'FAISS is a library for efficient similarity search and clustering of dense vectors.']
Python is a popular programming language known for its simplicity. T5 is a model developed by Google for various NLP tasks like summarization and Q&A. FAISS is a library for efficient similarity search and clustering of dense vectors.
Answer: Python is a popular programming language known for its simplicity.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch

# Step 1: Load a Pretrained Model and Tokenizer
model_name = "google/flan-t5-large"  # Example: GPT-2
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Set padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Step 2: Define LoRA Configuration
lora_config = LoraConfig(
    r=8,  # Low-rank matrices size
    lora_alpha=32,  # Scaling factor
    target_modules=["q", "v"],  # Modify query and value layers instead of attn.c_attn
    lora_dropout=0.1,  # Dropout to prevent overfitting
    bias="none"  # No additional bias parameters
)

# Step 3: Apply LoRA to the Model
lora_model = get_peft_model(model, lora_config)

# Print Model Summary
print("LoRA model initialized:")
lora_model.print_trainable_parameters()

# Step 4: Prepare Training Data
data = [
    "LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique.",
    "It adds small low-rank matrices to a large model during training.",
    "This approach reduces memory usage and computational cost.",
    "LoRA has been successfully applied to natural language processing tasks.",
    "Hugging Face provides tools to implement LoRA in Transformers."
]

# Tokenize
inputs = tokenizer(
    data,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=50
)

# Add attention mask
inputs["attention_mask"] = inputs["input_ids"].ne(tokenizer.pad_token_id)

# Step 5: Fine-Tune the LoRA Model
optimizer = torch.optim.AdamW(lora_model.parameters(), lr=1e-5)  # Lower learning rate
epochs = 20  # Increased number of epochs

for epoch in range(epochs):
    lora_model.train()
    outputs = lora_model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch + 1} - Loss: {loss.item()}")

# Step 6: Save the Fine-Tuned Model
lora_model.save_pretrained("lora_fine_tuned_model")

# Step 7: Inference with the Fine-Tuned Model
lora_model.eval()
prompt = "Tell me about LoRA."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
output_ids = lora_model.generate(
    input_ids,
    max_length=50,
    temperature=0.2,  # Control randomness
    top_k=5,  # Filter top K likely tokens
    top_p=0.5,  # Nucleus sampling
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
)
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("\nGenerated Text:")
print(generated_text)


LoRA model initialized:
trainable params: 2,359,296 || all params: 785,509,376 || trainable%: 0.3004


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 1 - Loss: 13.57359504699707
Epoch 2 - Loss: 12.905113220214844
Epoch 3 - Loss: 13.524273872375488
Epoch 4 - Loss: 13.83626651763916
Epoch 5 - Loss: 13.259330749511719
Epoch 6 - Loss: 13.400238037109375
Epoch 7 - Loss: 13.488359451293945
Epoch 8 - Loss: 13.866559028625488
Epoch 9 - Loss: 13.747057914733887
Epoch 10 - Loss: 13.28945255279541
Epoch 11 - Loss: 13.667774200439453
Epoch 12 - Loss: 13.178278923034668
Epoch 13 - Loss: 13.619804382324219
Epoch 14 - Loss: 13.903562545776367
Epoch 15 - Loss: 13.189322471618652
Epoch 16 - Loss: 13.4453125
Epoch 17 - Loss: 13.673174858093262
Epoch 18 - Loss: 13.563129425048828
Epoch 19 - Loss: 12.89649486541748
Epoch 20 - Loss: 13.436373710632324


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



Generated Text:
LoRA is a genus of fungi in the family Arctiinae .


In [ ]:
import os
import logging
import numpy as np
import ollama
from typing import List, Dict, Tuple
from dataclasses import dataclass
from sentence_transformers import SentenceTransformer
from pathlib import Path
import faiss

@dataclass
class MedicalDocument:
    content: str
    filename: str
    embedding: np.ndarray = None

class MedicalLlamaRAG:
    def __init__(self, embedding_model: str = 'all-MiniLM-L6-v2', llm_model: str = 'llama2'):
        self.documents: List[MedicalDocument] = []
        self.embedding_model = SentenceTransformer(embedding_model)
        self.llm_model = llm_model
        self.index = None


        try:
            ollama.pull(self.llm_model)
        except Exception as e:
            raise RuntimeError(f"Error initializing Llama 2: {str(e)}")

    def load_documents(self, folder_path: str) -> None:
        folder = Path(folder_path)
        if not folder.exists():
            raise FileNotFoundError(f"Folder not found: {folder_path}")

        for file_path in folder.glob("*.txt"):
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    self.documents.append(MedicalDocument(
                        content=f.read(),
                        filename=file_path.name
                    ))
            except Exception as e:
                continue

    def build_index(self) -> None:
        embeddings = []
        for doc in self.documents:
            embedding = self.embedding_model.encode(doc.content)
            doc.embedding = embedding
            embeddings.append(embedding)

        embeddings_array = np.vstack(embeddings)
        dimension = embeddings_array.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings_array)

    def get_relevant_docs(self, query: str, top_k: int = 3) -> List[Tuple[MedicalDocument, float]]:
        query_embedding = self.embedding_model.encode(query)
        distances, indices = self.index.search(
            query_embedding.reshape(1, -1),
            top_k
        )
        return [(self.documents[idx], dist) for idx, dist in zip(indices[0], distances[0])]

    def format_medical_prompt(self, query: str, relevant_docs: List[Tuple[MedicalDocument, float]]) -> str:
        context = "\n\n".join([
            f"{doc.content[:1000]}..."
            for doc, _ in relevant_docs
        ])

        return f"""Based on the following medical records, answer this question: {query}

Medical Records:
{context}

Answer based only on the information provided in these records."""

    def query(self, question: str, top_k: int = 3) -> str:
        if self.index is None:
            raise ValueError("Index not built. Call build_index() first.")

        relevant_docs = self.get_relevant_docs(question, top_k)
        prompt = self.format_medical_prompt(question, relevant_docs)

        try:
            response = ollama.chat(model=self.llm_model, messages=[
                {
                    "role": "system",
                    "content": "You are a medical information assistant. Provide answers based solely on the provided medical records."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ])
            return response['message']['content']
        except Exception as e:
            return f"Error generating response: {str(e)}"

def main():
    rag = MedicalLlamaRAG()
    rag.load_documents("C:\\Users\\shrih\\OneDrive\\Documents\\LLM\\Medical_reports")
    rag.build_index()

    # Simply print the response
    print(rag.query("What is the hospital name?"))

if __name__ == "__main__":
    main()